# Подключение LLM

## Задание 1. Первый запрос к модели через `/api/generate`

In [1]:
import requests

OLLAMA_URL = "http://localhost:11434"
LLM_MODEL = "qwen2.5:1.5b"


def generate_answer(prompt: str) -> str:
    """Отправляет промпт в Ollama и возвращает сгенерированный ответ."""

    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": LLM_MODEL,
            "prompt": prompt,
            "stream": False,
        },
        timeout=120,
    )
    response.raise_for_status()
    data = response.json()

    return data["response"].strip()


# Пробуем
answer = generate_answer(prompt="Что такое RAG в одном предложении?")
print(answer)


RAG stands for Retrieval-Augmented Generation. It is a type of machine learning model that combines two components: a retrieval system and a generation model. This architecture is designed to improve the accuracy and quality of machine-generated content, such as text or summaries, by leveraging pre-existing knowledge and information from a knowledge base.


## Задание 2. Построение RAG промпта

In [ ]:
PROMPT_TEMPLATE = """Ты ассистент по документации Ollama.
Отвечай на вопрос пользователя, опираясь только на приведённые ниже фрагменты документации.
Если ответа во фрагментах нет - честно скажи, что не знаешь.
Ответ давай на русском языке.
В конце ответа на отдельной строке укажи источник в формате: Источник: <имя файла>.

Фрагменты документации:
{context}

Вопрос пользователя: {question}

Ответ:"""


def build_prompt(question: str, retrieved: list[dict]) -> str:
    """Собирает промпт из системной инструкции, контекста и вопроса."""
    context_parts = []
    for r in retrieved:
        context_parts.append(f"[файл: {r['source']}]\n{r['text']}")
    context = "\n\n---\n\n".join(context_parts)
    return PROMPT_TEMPLATE.format(context=context, question=question)

In [ ]:
# Чанкинг, эмбеддинги и FAISS-индекс живут в rag.py - тот же код, что в embeddings.ipynb
from rag import build_index, vector_search

rag = build_index("docs")
print(f"Чанков: {len(rag.chunks)}, векторов в индексе: {rag.index.ntotal}")
